# Fine-tuning Llama 3.2 3B on Seinfeld Scripts with QLoRA

This notebook walks through fine-tuning **Llama 3.2 3B** on ~2,300 Seinfeld script excerpts using **QLoRA** — LoRA on top of a 4-bit quantized base model. The result is a model that generates high-quality Seinfeld dialogue entirely in the browser.

## What you'll learn
1. How QLoRA extends LoRA with 4-bit quantization
2. How to fine-tune Llama 3.2 3B on a single GPU
3. How to convert to GGUF for browser deployment via wllama/llama.cpp

## Requirements
- **GPU**: A100 40GB or 80GB (Colab Pro/Pro+)
- **Time**: ~25 minutes for 5 epochs
- **HuggingFace account**: with access to `meta-llama/Llama-3.2-3B` (request at [huggingface.co/meta-llama](https://huggingface.co/meta-llama/Llama-3.2-3B))
- **HF_TOKEN**: set in Colab Secrets (key icon in left sidebar)

## QLoRA: LoRA on a 4-bit Base Model

### The memory problem

Llama 3.2 3B has **3.2 billion parameters**. In fp16 that's ~6.4 GB just for the model weights — plus optimizer states, gradients, and activations, you need ~24+ GB to fine-tune. Even LoRA (which only trains small adapter matrices) still needs to load the full base model in fp16.

### QLoRA's solution

[QLoRA (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314) combines three techniques:

**1. 4-bit NormalFloat (NF4) quantization**

The base model weights are quantized to 4 bits using NF4 — a data type optimized for normally-distributed neural network weights. Each weight is mapped to one of 16 values, reducing the model from ~6.4 GB (fp16) to ~1.6 GB (4-bit).

```
fp16:  2 bytes per weight × 3.2B = 6.4 GB
NF4:   0.5 bytes per weight × 3.2B = 1.6 GB  (4x smaller)
```

**2. Double quantization**

The quantization constants themselves (one per block of 64 weights) are further quantized from fp32 to fp8, saving an additional ~0.37 bits per parameter.

**3. LoRA adapters in bf16**

The trainable LoRA matrices ($B$ and $A$) are kept in bfloat16 for full precision during training. During the forward pass, 4-bit base weights are dequantized to bf16 on-the-fly, combined with LoRA outputs, and the result flows through the network.

$$\text{output} = W_{\text{4bit}}^{\text{(dequant)}} \cdot x + \frac{\alpha}{r} \cdot B \cdot A \cdot x$$

### What we target

For Llama models, we apply LoRA to **all linear layers** in the transformer:

| Module | Role | Count per layer |
|--------|------|-----------------|
| `q_proj`, `k_proj`, `v_proj` | Attention: query, key, value projections | 3 |
| `o_proj` | Attention: output projection | 1 |
| `gate_proj`, `up_proj`, `down_proj` | MLP (SwiGLU feed-forward) | 3 |

That's **7 modules × 26 layers = 182 LoRA pairs**, with rank $r = 32$ and $\alpha = 64$.

Total trainable parameters: **~51.4M** out of 3.2B (**1.6%**).

### LoRA rank: how to choose

| Rank | Trainable % | Use case |
|------|-------------|----------|
| 8 | ~0.4% | Simple style transfer, light format changes |
| 16 | ~0.8% | Format learning + moderate content shift |
| 32 | ~1.6% | Our choice — format + character voices + topic following |
| 64 | ~3.2% | Heavy adaptation (e.g. new language, domain shift) |

Higher rank = more expressive but slower training and more memory. For teaching a 3B model a specific dialogue format, $r = 32$ is the sweet spot.

---
## 0. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/detrin/gpt-seinfeld /content/gpt-seinfeld
%cd /content/gpt-seinfeld

In [ ]:
# Install dependencies
!pip install -q "transformers>=4.44" "peft>=0.17.0" "bitsandbytes>=0.43.0" datasets accelerate torch

In [ ]:
# HuggingFace login — REQUIRED for Llama (gated model)
import os

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError(
        "HF_TOKEN not found in Colab Secrets. "
        "Go to huggingface.co/settings/tokens, create a token, "
        "then add it as HF_TOKEN in Colab Secrets (key icon in left sidebar)."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)
print("HF login OK")

In [ ]:
# Mount Google Drive for saving checkpoints
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!nvidia-smi | head -5

---
## 1. Load the dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset = load_dataset("json", data_files="data/processed/train_v2.jsonl", split="train")


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Dataset: {len(tokenized)} examples")

---
## 2. Load model in 4-bit and attach LoRA adapters

This is the core of QLoRA: load the base model quantized to 4-bit NF4, then attach trainable LoRA adapters in bf16.

In [ ]:
import torch
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # quantize base weights to 4-bit
    bnb_4bit_use_double_quant=True,  # quantize the quantization constants too
    bnb_4bit_quant_type="nf4",  # NormalFloat4 — optimal for neural net weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16 during forward pass
)

# Load base model in 4-bit
print("Loading Llama 3.2 3B in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare for k-bit training (handles gradient checkpointing, layer norms in fp32, etc.)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded. Memory: {model.get_memory_footprint() / 1024**3:.1f} GB")

In [ ]:
# Attach LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,  # rank — 32 is the sweet spot for 3B models
    lora_alpha=64,  # scaling: alpha/r = 2.0
    lora_dropout=0.05,
    target_modules=[  # all linear layers in the transformer
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",  # attention
        "gate_proj",
        "up_proj",
        "down_proj",  # MLP (SwiGLU)
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~51.4M trainable out of 3.2B total (1.6%)

---
## 3. Train

Key training settings:
- **5 epochs** with effective batch size 16 (batch=4 × grad_accum=4)
- **Gradient checkpointing** enabled — trades compute for memory, essential for fitting on a single GPU
- **bf16 mixed precision** — Llama was trained in bf16, so we match it

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

CHECKPOINT_DIR = "/content/drive/MyDrive/llama-seinfeld"

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size = 16
    warmup_steps=100,
    learning_rate=2e-4,
    weight_decay=0.01,
    bf16=True,  # match Llama's training dtype
    gradient_checkpointing=True,  # save memory at cost of ~20% slower training
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

trainer.train()

---
## 4. Merge and save

In [ ]:
# Merge LoRA into base weights
merged = model.merge_and_unload()

if hasattr(merged, "gradient_checkpointing_disable"):
    merged.gradient_checkpointing_disable()
merged.config.use_cache = True

# Save in fp16 (the merged model is dequantized back to fp16 during merge)
merged.save_pretrained(CHECKPOINT_DIR, safe_serialization=True)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Merged model saved to {CHECKPOINT_DIR}")

---
## 5. Sanity check

In [ ]:
from transformers import pipeline

# Free training memory
del model, merged, trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

gen = pipeline(
    "text-generation", model=CHECKPOINT_DIR, device_map="auto", torch_dtype=torch.bfloat16
)

topics = ["losing a parking spot", "double dipping", "the contest"]
for topic in topics:
    prompt = f"TOPIC: {topic}\n\nCHARACTERS: JERRY, GEORGE, ELAINE, KRAMER\n\n["
    out = gen(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_k=8,
        repetition_penalty=1.15,
    )
    text = out[0]["generated_text"][len(prompt) - 1 :]  # keep the '['
    print(f"{'=' * 60}\nTOPIC: {topic}\n{'=' * 60}")
    print(text[:500])
    print()

---
## 6. Convert to GGUF for browser deployment

To run in the browser via [wllama](https://github.com/ngxson/wllama) (WebAssembly port of llama.cpp), we need to convert the model to **GGUF format** and quantize to 4-bit.

### Why GGUF?

GGUF is llama.cpp's native format. It stores weights in quantized form with metadata, and is designed for fast loading and inference. The 4-bit quantized GGUF is ~1.9 GB — small enough to download in a browser.

### Quantization: Q4_K_M

We use `Q4_K_M` (4-bit with medium quality K-quant):
- Attention and MLP weights: 4-bit with per-group scaling
- Important layers (first/last, attention output): kept at higher precision
- Result: ~1.9 GB, excellent quality/size tradeoff

| Format | Size | Quality | Use case |
|--------|------|---------|----------|
| fp16 | ~6.4 GB | Best | Server deployment |
| Q8_0 | ~3.4 GB | Near-lossless | Desktop inference |
| Q4_K_M | ~1.9 GB | Good | Browser / mobile |
| Q4_0 | ~1.8 GB | OK | Smallest practical size |

In [ ]:
# Install llama.cpp conversion tools
!pip install -q llama-cpp-python
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
GGUF_DIR = "/content/llama-seinfeld-gguf"

# Step 1: Convert HF safetensors → GGUF fp16
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {CHECKPOINT_DIR} \
    --outfile {GGUF_DIR}/seinfeld-3b-f16.gguf \
    --outtype f16

print("fp16 GGUF created")

In [ ]:
# Step 2: Quantize to Q4_K_M
# Build llama-quantize (requires cmake)
!cd /content/llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j$(nproc)

!mkdir -p {GGUF_DIR}
!/content/llama.cpp/build/bin/llama-quantize \
    {GGUF_DIR}/seinfeld-3b-f16.gguf \
    {GGUF_DIR}/seinfeld-3b-q4_k_m.gguf \
    Q4_K_M

import os

f16_size = os.path.getsize(f"{GGUF_DIR}/seinfeld-3b-f16.gguf") / 1024**3
q4_size = os.path.getsize(f"{GGUF_DIR}/seinfeld-3b-q4_k_m.gguf") / 1024**3
print(f"fp16: {f16_size:.1f} GB → Q4_K_M: {q4_size:.1f} GB ({q4_size / f16_size:.0%} of original)")

### Sharding for browser memory limits

Browsers can struggle with single files >500 MB. We shard the GGUF into ~512 MB chunks so wllama can load them sequentially.

In [ ]:
# Step 3: Shard into ~512 MB chunks for browser loading
!split -b 512m {GGUF_DIR}/seinfeld-3b-q4_k_m.gguf {GGUF_DIR}/seinfeld-3b-q4_k_m-

# Rename to GGUF shard convention
import glob

shards = sorted(glob.glob(f"{GGUF_DIR}/seinfeld-3b-q4_k_m-*"))
total = len(shards)
for i, shard in enumerate(shards):
    new_name = f"{GGUF_DIR}/seinfeld-3b-q4_k_m-{i + 1:05d}-of-{total:05d}.gguf"
    os.rename(shard, new_name)
    mb = os.path.getsize(new_name) / 1024**2
    print(f"{os.path.basename(new_name)}: {mb:.0f} MB")

print(f"\n{total} shards ready for upload to HuggingFace.")

---
## 7. Upload to HuggingFace Hub (optional)

In [ ]:
# Uncomment and fill in to push:
#
# from huggingface_hub import HfApi
# api = HfApi()
# for shard in sorted(glob.glob(f'{GGUF_DIR}/*-of-*.gguf')):
#     api.upload_file(
#         path_or_fileobj=shard,
#         path_in_repo=os.path.basename(shard),
#         repo_id='YOUR_USERNAME/Llama-3.2-3B-Seinfeld-GGUF',
#         repo_type='model',
#     )
#     print(f'Uploaded {os.path.basename(shard)}')